# Reinforcement Learning — Lab 4

## CartPole with REINFORCE

**Goal:** learn a stochastic policy directly from complete episodes using the REINFORCE policy-gradient algorithm.

You will:
1. explore CartPole and the Gymnasium interaction loop;
2. represent a policy with a neural network and sample actions from it;
3. compute discounted returns and implement REINFORCE;
4. evaluate the learned policy without exploration; and
5. compare learning with and without a value-function baseline.

The policy does not choose the action with the largest score. It produces a probability distribution and samples an action. REINFORCE increases the probability of actions that led to better-than-expected outcomes.


### Suggested submission

- Your completed notebook, with TODOs filled in
- The learning curves for REINFORCE with and without a baseline
- Brief answers to the questions in **Part 8**


---

## Part 1 — Setup

This notebook runs in Google Colab or a local Python environment. It uses PyTorch for the policy and Gymnasium's CartPole environment. No rendering is required for training.


In [ ]:
!pip -q install "gymnasium[classic-control]"

import random
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import torch
from torch import nn
from torch.distributions import Categorical

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)


## Part 2 — Meet CartPole

The observation is a vector containing cart position, cart velocity, pole angle, and pole angular velocity. The two actions push the cart left or right. The reward is +1 for each step the pole remains upright, so longer episodes have larger returns. CartPole-v1 ends when the pole falls or after 500 steps.

Gymnasium's `step` returns both `terminated` and `truncated`. We will end a sampled trajectory on either signal. Since REINFORCE uses the observed episode return, it does not bootstrap a value at that boundary.


In [ ]:
env = gym.make("CartPole-v1")
obs, info = env.reset(seed=SEED)
print("Observation:", obs)
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
print("One step:", reward, terminated, truncated)
env.close()


## Part 3 — A stochastic policy

A neural network maps the observation to one score (logit) per action. Softmax turns the scores into action probabilities. We sample from this categorical distribution so that the policy remains stochastic.

The log probability of the sampled action is what the policy-gradient update needs. PyTorch's distribution object provides both the sampling and log probability calculations.


In [ ]:
class Policy(nn.Module):
    def __init__(self, observation_dim, action_count):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(observation_dim, 64),
            nn.Tanh(),
            nn.Linear(64, action_count),
        )

    def forward(self, observation):
        return self.network(observation)


def sample_action(policy, observation):
    """Return a sampled action and its log probability."""
    state = torch.as_tensor(observation, dtype=torch.float32, device=device)
    logits = policy(state)
    distribution = Categorical(logits=logits)
    action = distribution.sample()
    return int(action.item()), distribution.log_prob(action)


Try sampling several actions from the untrained policy. The probabilities need not be exactly equal, but both actions should be possible.


In [ ]:
probe_env = gym.make("CartPole-v1")
policy = Policy(probe_env.observation_space.shape[0], probe_env.action_space.n).to(device)
probe_obs, _ = probe_env.reset(seed=SEED)
with torch.no_grad():
    probs = torch.softmax(policy(torch.tensor(probe_obs, dtype=torch.float32, device=device)), dim=-1)
print("Action probabilities:", probs.cpu().numpy())
probe_env.close()


## Part 4 — Collect one episode

REINFORCE updates from complete trajectories. For each time step, store the observation, sampled action's log probability, and reward. The log probabilities retain their computation graph so that gradients can flow back into the policy after the episode ends.


In [ ]:
def collect_episode(env, policy, seed=None):
    observation, _ = env.reset(seed=seed)
    log_probs, rewards, states = [], [], []
    terminated = truncated = False

    while not (terminated or truncated):
        states.append(torch.as_tensor(observation, dtype=torch.float32, device=device))
        action, log_prob = sample_action(policy, observation)
        observation, reward, terminated, truncated, _ = env.step(action)
        log_probs.append(log_prob)
        rewards.append(float(reward))

    return states, log_probs, rewards

rollout_env = gym.make("CartPole-v1")
_, _, demo_rewards = collect_episode(rollout_env, policy, seed=SEED)
print("Episode length:", len(demo_rewards), " total reward:", sum(demo_rewards))
rollout_env.close()


## Part 5 — Discounted returns

The return from time *t* is the sum of future rewards, discounted by γ:

\[G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots\]

Compute returns by working backwards through the episode. Scale returns by their within-episode standard deviation to keep update magnitudes more consistent. We do not subtract the episode mean here, so the vanilla REINFORCE run has no added return baseline.


In [ ]:
def discounted_returns(rewards, gamma=0.99):
    returns = []
    running_return = 0.0
    for reward in reversed(rewards):
        # TODO: update running_return using this reward and gamma,
        # then add the result to returns.
        pass
    returns.reverse()
    returns = torch.tensor(returns, dtype=torch.float32, device=device)
    if len(returns) > 1:
        # Scale the update while preserving the return baseline for comparison.
        returns = returns / (returns.std(unbiased=False) + 1e-8)
    return returns

print("Example:", discounted_returns([1, 1, 1], gamma=0.9).cpu().numpy())


## Part 6 — REINFORCE update

For a sampled trajectory, the policy objective is to increase the log probability of actions followed by high returns. Gradient ascent on \(\log \pi(A_t|S_t)G_t\) is implemented as gradient descent on its negative:

\[L(	heta) = -\sum_t \log \pi_	heta(A_t|S_t)G_t.\]

Complete the update. The optimizer performs gradient descent, so keep the leading minus sign.


In [ ]:
def reinforce_update(policy, optimizer, log_probs, returns):
    # TODO: form the REINFORCE loss from log_probs and returns.
    # TODO: clear old gradients, backpropagate, and take one optimizer step.
    pass


### Train vanilla REINFORCE

We will report the average episode length over a moving window. A score near 500 means the policy often keeps the pole upright until the time limit. Learning can vary with the random seed and hyperparameters.


In [ ]:
def train_reinforce(episodes=700, learning_rate=1e-2, gamma=0.99, seed=SEED):
    env = gym.make("CartPole-v1")
    env.action_space.seed(seed)
    torch.manual_seed(seed)
    agent = Policy(env.observation_space.shape[0], env.action_space.n).to(device)
    optimizer = torch.optim.Adam(agent.parameters(), lr=learning_rate)
    episode_lengths = []

    for episode in range(episodes):
        states, log_probs, rewards = collect_episode(env, agent, seed=seed + episode)
        returns = discounted_returns(rewards, gamma)
        reinforce_update(agent, optimizer, log_probs, returns)
        episode_lengths.append(len(rewards))

        if (episode + 1) % 50 == 0:
            recent = np.mean(episode_lengths[-50:])
            print(f"Episode {episode + 1:4d} | mean length (last 50): {recent:6.1f}")

    env.close()
    return agent, episode_lengths

# Run after completing the TODOs.
# reinforce_policy, reinforce_lengths = train_reinforce()


### Plot the training curve

The curve is noisy because each episode is sampled from the current stochastic policy. The moving average helps show the trend.


In [ ]:
def moving_average(values, window=25):
    return np.convolve(values, np.ones(window) / window, mode="valid")

# Uncomment after training:
# plt.plot(moving_average(reinforce_lengths))
# plt.xlabel("Episode")
# plt.ylabel("Mean episode length (25 episode window)")
# plt.title("CartPole: vanilla REINFORCE")
# plt.ylim(0, 500)
# plt.grid(alpha=0.3)
# plt.show()


## Part 7 — Add a value baseline

Returns can vary greatly between episodes. A state-value estimate provides a baseline: reinforce an action when its return is above the predicted value, and reduce its probability when it is below. The advantage estimate is \(G_t - V_\phi(S_t)\).

The baseline changes the variance of the policy-gradient estimate, while leaving its expected direction unchanged. Train the critic to predict the observed returns. Detach the advantage in the actor loss so that the policy update does not train the value network.


In [ ]:
class ValueFunction(nn.Module):
    def __init__(self, observation_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(observation_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )

    def forward(self, states):
        return self.network(states).squeeze(-1)


def actor_critic_episode_update(policy, value_function, actor_optimizer,
                                critic_optimizer, states, log_probs, returns):
    state_batch = torch.stack(states)
    values = value_function(state_batch)
    advantages = returns - values.detach()

    # TODO: construct actor loss using log_probs and advantages.
    # TODO: construct critic loss between values and returns.
    # TODO: update actor and critic with their separate optimizers.
    pass


### Train REINFORCE with a learned baseline

This remains an episodic REINFORCE actor update. The critic supplies the baseline; it does not replace the sampled return with a one-step TD target.


In [ ]:
def train_reinforce_baseline(episodes=700, actor_lr=1e-2, critic_lr=1e-3,
                             gamma=0.99, seed=SEED):
    env = gym.make("CartPole-v1")
    env.action_space.seed(seed)
    torch.manual_seed(seed)
    policy = Policy(env.observation_space.shape[0], env.action_space.n).to(device)
    value_function = ValueFunction(env.observation_space.shape[0]).to(device)
    actor_optimizer = torch.optim.Adam(policy.parameters(), lr=actor_lr)
    critic_optimizer = torch.optim.Adam(value_function.parameters(), lr=critic_lr)
    episode_lengths = []

    for episode in range(episodes):
        states, log_probs, rewards = collect_episode(env, policy, seed=seed + episode)
        returns = discounted_returns(rewards, gamma)
        actor_critic_episode_update(policy, value_function, actor_optimizer,
                                   critic_optimizer, states, log_probs, returns)
        episode_lengths.append(len(rewards))

        if (episode + 1) % 50 == 0:
            recent = np.mean(episode_lengths[-50:])
            print(f"Episode {episode + 1:4d} | mean length (last 50): {recent:6.1f}")

    env.close()
    return policy, value_function, episode_lengths

# Run after completing the actor and critic update TODOs.
# baseline_policy, value_model, baseline_lengths = train_reinforce_baseline()


### Compare learning curves

Run each method with the same number of episodes and seed. To get a more reliable comparison, repeat with several seeds and compare the mean and spread of the final scores.


In [ ]:
# Uncomment after both agents have been trained:
# plt.plot(moving_average(reinforce_lengths), label="REINFORCE")
# plt.plot(moving_average(baseline_lengths), label="REINFORCE + value baseline")
# plt.xlabel("Episode")
# plt.ylabel("Mean episode length (25 episode window)")
# plt.title("CartPole policy-gradient comparison")
# plt.ylim(0, 500)
# plt.legend()
# plt.grid(alpha=0.3)
# plt.show()


## Part 8 — Evaluate a learned policy

Training performance includes variability from sampling. Evaluate the policy over fresh episodes and report the mean and standard deviation of episode length.


In [ ]:
def evaluate_policy(policy, episodes=50, seed=10000):
    env = gym.make("CartPole-v1")
    lengths = []
    for episode in range(episodes):
        observation, _ = env.reset(seed=seed + episode)
        terminated = truncated = False
        length = 0
        while not (terminated or truncated):
            with torch.no_grad():
                state = torch.as_tensor(observation, dtype=torch.float32, device=device)
                logits = policy(state)
                action = int(torch.argmax(logits).item())  # deterministic evaluation
            observation, _, terminated, truncated, _ = env.step(action)
            length += 1
        lengths.append(length)
    env.close()
    return np.asarray(lengths)

# Example after training:
# scores = evaluate_policy(reinforce_policy)
# print(f"Mean length: {scores.mean():.1f} ± {scores.std():.1f}")


## Questions

1. What does the policy network output, and why do we sample an action during training?
2. In the REINFORCE loss, what role does each return \(G_t\) play?
3. Why does this implementation wait until the episode ends before updating the policy?
4. How does the value baseline change the policy update? Why do we detach the advantage in the actor loss?
5. Compare the learning curves. Did the baseline make learning faster or less variable in your run? What evidence supports your answer?
6. How might results change if you alter the learning rate, discount factor, or network size?
7. Why do we use a deterministic action for evaluation even though the learned policy is stochastic?

### Answers

1.  
2.  
3.  
4.  
5.  
6.  
7.  

## Optional extensions

1. Repeat both experiments across at least five seeds and plot mean performance with uncertainty.
2. Compare normalized and unnormalized returns.
3. Plot the policy's probability of pushing right over a grid of cart positions and pole angles, holding velocities near zero.
4. Change the policy network to use a `Softmax` output and compare with passing logits directly to `Categorical`.
5. Investigate how treating the 500-step time limit as an episode boundary affects return estimates.
